# 4.1 Variational Autoencoder (I): Training
***

## General configuration

In [ ]:
### Configuration ###
config = {
    'BATCH_SIZE':    16,
    'LATENT_DIM':    16,
    'LEARNING_RATE': 1E-3,
    'NUM_EPOCHS':    400,
    'BETA':          2.0,
    'RANDOM_SEED':   23,
    'FNAME_MODEL':   'vae_L16_c.pt',
    }

## Main functions

#### Importing modules

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchsummary import summary

#### Training function

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    # Set training mode
    model.train()
    
    total_loss  = 0.0
    total_recon = 0.0
    total_kl    = 0.0
    num_batches = 0
    
    for batch in loader:
        # Model prediction
        prediction, mu, logvar = model(batch)
        # Compute loss
        loss, recon, kl = criterion(prediction,batch,mu,logvar)

        # Update weight
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Update metrics
        total_loss  += loss.item()
        total_recon += recon.item()
        total_kl    += kl.item()
        num_batches += 1
    return {'loss': total_loss/num_batches, 
            'recon': total_recon/num_batches,
            'kl': total_kl/num_batches}

#### Evaluation function

In [ ]:
def evaluate_epoch(model, loader, criterion):
    # Set inference mode
    model.eval()
    
    total_loss  = 0.0
    total_recon = 0.0
    total_kl    = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in loader:
            # Model prediction
            prediction, mu, logvar = model(batch)
            # Compute loss
            loss, recon, kl = criterion(prediction,batch,mu,logvar)

            # Update metrics
            total_loss  += loss.item()
            total_recon += recon.item()
            total_kl    += kl.item()
            num_batches += 1
    return {'loss': total_loss/num_batches, 
            'recon': total_recon/num_batches,
            'kl': total_kl/num_batches}

## 1. Loading raw data and normaliation

In [ ]:
from os.path import isfile

## Get a FALL3D ensemble run output
fname = "data/tephra_col_mass.ens.nc"
if not isfile(fname):
    !wget -P ./data https://saco.csic.es/s/wFpKYG5bHfTwKbi/download/tephra_col_mass.ens.nc

In [ ]:
import xarray as xr

ds = xr.open_dataset(fname)
da = ds["tephra_col_mass"]

In [ ]:
from helper import MinMaxScale

## Re-scale between 0 and 20
## so 98% of the data is between 0 and 1
min_value = 0
max_value = 20
transform = MinMaxScale(min_value, max_value)

## 2. Create a custom Dataset and splitting

In [ ]:
from helper import EnsembleDataset

## Create a Dataset object for the full dataset (training + validation)
dataset = EnsembleDataset(da, transform)

## Random split with in training and validation datasets
n_total = len(dataset)
n_train = int(0.8 * n_total)   # 80% train
n_val   = n_total - n_train    # 20% val

#torch.manual_seed(config['RANDOM_SEED'])
train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

## 3. Create a DataLoader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=config['BATCH_SIZE'], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config['BATCH_SIZE'], shuffle=False)

In [ ]:
## Load data by mini-batches with dimensions:
## (nbatch,nchannels,nlat,nlon)
for batch in train_loader:
    print("Batch dimensions: (nbatch,nchannels,nlat,nlon)")
    print(batch.shape)
    break

## 4. Define a model

In [ ]:
from helper import VariationalAutoencoder

model = VariationalAutoencoder(config['LATENT_DIM'])
summary(model, (1,101,121))

## 5. Loss function

In [ ]:
from helper import VAELoss

criterion = VAELoss(beta=config['BETA'])

## 6. Optimizer

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=config['LEARNING_RATE'])

## Training loop

In [ ]:
# Evaluation metrics for every  epoch
train_losses = []
val_losses   = []

for epoch in range(config['NUM_EPOCHS']):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss   = evaluate_epoch(model, val_loader, criterion)
    # Store current losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if epoch%10 == 0 or epoch == config['NUM_EPOCHS']-1:
        print(f"-> Epoch {epoch+1:02d} \n"
              f"   Train loss {train_loss} \n"
              f"   Validation loss: {val_loss}")
print("Done!")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axs = plt.subplots(nrows=2, sharex=True)

axs[0].set(ylabel='Reconstruction loss', yscale='log')
axs[1].set(ylabel='KL divergence', xlabel='Epoch')

df_train = pd.DataFrame(train_losses)
df_val   = pd.DataFrame(val_losses)

df_train.plot(y='recon', 
              label='Training', 
              ax=axs[0])

df_val.plot(y='recon',
            label='Validation',
            ax=axs[0])

df_train.plot(y='kl',
            label='Training',
            ax=axs[1])

df_val.plot(y='kl',
            label='Validation',
            ax=axs[1])

## Save trained model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),  # Trained Model parameters
    'LATENT_DIM': config['LATENT_DIM'],      # Dimension of the latent space (=2)
    'MINVAL': min_value,                     # Min value used for normalization (=0)
    'MAXVAL': max_value,                     # Max value used for normalization (=20)
    'LOSS':  df_val['loss'].iloc[-1].item(), 
    'RECON': df_val['recon'].iloc[-1].item(), 
    'KL':    df_val['kl'].iloc[-1].item(), 
    }, config['FNAME_MODEL'])

## State reconstruction

In [ ]:
## Reconstruct the validation dataset
xp_list = []
xv_list = []
with torch.no_grad():
    for batch in val_loader:
        prediction, _, _ = model(batch)
        xp = prediction.squeeze(1)
        xv = batch.squeeze(1)
        xp_list.append(transform.invert(xp))
        xv_list.append(transform.invert(xv))
    xp = torch.cat(xp_list, dim=0)
    xv = torch.cat(xv_list, dim=0)

In [ ]:
## Plotting reconstructions for validation dataset 
plot_conf = {
    'cmap': 'RdYlBu_r',
    'vmin': 0, 
    'vmax': 30,
}

n=min(12,n_val)
fig, axs = plt.subplots(nrows = n, ncols = 2, figsize=(6,38))

for i in range(n):
    cs1=axs[i,0].pcolormesh(da.lon,da.lat,xv[i], **plot_conf)
    cs2=axs[i,1].pcolormesh(da.lon,da.lat,xp[i], **plot_conf)
                        
for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    
cbar = fig.colorbar(cs2, 
             ax=axs, 
             orientation='horizontal',
             fraction=0.05,
             pad=0.02, 
             aspect=30
            )
cbar.set_label('Column mass [g/m2]')

## Latent Space Visualization

In [ ]:
fig, ax = plt.subplots()

with torch.no_grad():
    for batch in train_loader:
        mu, logvar = model.encode(batch)
        z = model.reparameterize(mu,logvar)
        ax.scatter(z[:,0],z[:,1], color='red')
    for batch in val_loader:
        mu, logvar = model.encode(batch)
        z = model.reparameterize(mu,logvar)
        ax.scatter(z[:,0],z[:,1], color='blue')